In [1]:
import torch
import pandas as pd
from typing import List, Dict, Set
# NOTE: gensim is not available due to compatibility issues with Python 3.13
# We'll create a simplified version for demonstration

def load_sst2_data(file_path: str) -> pd.DataFrame:
    """
    SST-2のデータを読み込む

    Args:
        file_path (str): データファイルのパス

    Returns:
        pd.DataFrame: 読み込んだデータ
    """
    return pd.read_csv(file_path, sep="\t", header=0)


def get_vocabulary(df: pd.DataFrame) -> Set[str]:
    """
    データセットに含まれる単語の集合を取得する

    Args:
        df (pd.DataFrame): データセット

    Returns:
        Set[str]: 単語の集合
    """
    vocabulary = set()
    for text in df["sentence"]:
        vocabulary.update(text.lower().split())
    return vocabulary


def create_word_to_id_mapping(vocabulary: Set[str]) -> Dict[str, int]:
    """
    単語からIDへのマッピングを作成する（gensim代替）

    Args:
        vocabulary (Set[str]): 必要な単語の集合

    Returns:
        Dict[str, int]: 単語からIDへの辞書
    """
    # 単語からIDへの辞書を作成
    word_to_id = {"<PAD>": 0}  # パディングトークンのIDは0
    
    # 語彙の単語にIDを割り当て
    for word in sorted(vocabulary):  # 一貫性のためソート
        word_to_id[word] = len(word_to_id)

    return word_to_id


def convert_text_to_ids(text: str, word_to_id: Dict[str, int]) -> List[int]:
    """
    テキストをトークンID列に変換する

    Args:
        text (str): 変換するテキスト
        word_to_id (Dict[str, int]): 単語からIDへの辞書

    Returns:
        List[int]: トークンID列
    """
    # テキストを小文字に変換し、トークン化
    tokens = text.lower().split()

    # 語彙に含まれるトークンのIDのみを取得
    ids = [word_to_id[token] for token in tokens if token in word_to_id]

    return ids


def process_sst2_data(file_path: str, word_to_id: Dict[str, int]) -> List[Dict]:
    """
    SST-2のデータを処理し、トークンID列に変換する

    Args:
        file_path (str): データファイルのパス
        word_to_id (Dict[str, int]): 単語からIDへの辞書

    Returns:
        List[Dict]: 処理されたデータ
    """
    # データを読み込む
    df = load_sst2_data(file_path)
    return process_sst2_data_from_df(df, word_to_id)


def process_sst2_data_from_df(df: pd.DataFrame, word_to_id: Dict[str, int]) -> List[Dict]:
    """
    DataFrameからSST-2のデータを処理し、トークンID列に変換する

    Args:
        df (pd.DataFrame): データフレーム
        word_to_id (Dict[str, int]): 単語からIDへの辞書

    Returns:
        List[Dict]: 処理されたデータ
    """
    processed_data = []

    for _, row in df.iterrows():
        # テキストをトークンID列に変換
        input_ids = convert_text_to_ids(row["sentence"], word_to_id)

        # 空のトークン列の場合はスキップ
        if not input_ids:
            continue

        # データを辞書形式で保存
        data = {
            "text": row["sentence"],
            "label": torch.tensor([float(row["label"])]),
            "input_ids": torch.tensor(input_ids),
        }
        processed_data.append(data)

    return processed_data


# 訓練データと開発データを読み込む
train_df = load_sst2_data("../chapter07/SST-2/train.tsv")
dev_df = load_sst2_data("../chapter07/SST-2/dev.tsv")

# 必要な単語の集合を取得
vocabulary = get_vocabulary(train_df)
vocabulary.update(get_vocabulary(dev_df))

print(f"語彙サイズ: {len(vocabulary)}")
print(f"語彙の例: {list(vocabulary)[:10]}")

# 単語からIDへのマッピングを作成
word_to_id = create_word_to_id_mapping(vocabulary)

print(f"単語->IDマッピングの例:")
for word, id in list(word_to_id.items())[:10]:
    print(f"  {word}: {id}")

# 訓練データを処理
train_data = process_sst2_data_from_df(train_df, word_to_id)
print(f"訓練データ数: {len(train_data)}")

# 開発データを処理
dev_data = process_sst2_data_from_df(dev_df, word_to_id)
print(f"開発データ数: {len(dev_data)}")

# サンプルを表示
print("\n訓練データのサンプル:")
sample = train_data[0]
print(f"テキスト: {sample['text']}")
print(f"ラベル: {sample['label']}")
print(f"トークンID列: {sample['input_ids']}")

語彙サイズ: 15756
語彙の例: ['1975', 'robbery', 'unrecoverable', 'hormonal', 'afford', 'affectionate', 'empathy', 'free', 'struggling', 'relationship']
単語->IDマッピングの例:
  <PAD>: 0
  !: 1
  !?: 2
  #: 3
  $: 4
  %: 5
  &: 6
  ': 7
  '': 8
  '30s: 9
訓練データ数: 67349
開発データ数: 872

訓練データのサンプル:
テキスト: hide new secretions from the parental units 
ラベル: tensor([0.])
トークンID列: tensor([ 6512,  9308, 12100,  5615, 13952,  9947, 14777])
